In [2]:
import os

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [1]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

# --- YOUR CODE HERE ---

In [3]:
documents = [
    Document(
        page_content=(
            "LangGraph is a framework for building stateful "
            "and multi-agent AI applications."
        ),
        metadata={"source": "langgraph_notes"},
    ),
       Document(
        page_content=(
            "Inception BD is a Edtech Platform "
            "and provides courses on AI."
        ),
        metadata={"source": "inception_notes"},
    ),
    Document(
        page_content=(
            "RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information before generating an answer."
        ),
        metadata={"source": "rag_notes"},
    ),
    Document(
        page_content=(
            "Groq provides fast inference for supported large "
            "language models through the Groq API."
        ),
        metadata={"source": "groq_notes"},
    ),
    Document(
        page_content=(
            "FAISS is a vector similarity-search library. "
            "It can retrieve documents whose embeddings are close "
            "to the query embedding."
        ),
        metadata={"source": "faiss_notes"},
    ),
]


In [4]:
type(documents[0])

langchain_core.documents.base.Document

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True,
    },
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8961.26it/s]


In [6]:
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)


In [7]:
vector_store.save_local("faiss_index")

In [8]:
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)


In [9]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

In [22]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_retries=2,
    api_key = 'GROQ_API_KEY'
)


In [13]:
response = llm.invoke("Hi Tell me about you")

In [14]:
response.content

'I\'m an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."'

In [15]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the question using only the provided context.

If the answer is not present in the context, say:
"I do not know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [16]:
def format_documents(retrieved_documents: list[Document]) -> str:
    return "\n\n".join(
        document.page_content
        for document in retrieved_documents
    )

In [17]:
rag_chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [18]:
def ask_question(question: str) -> str:
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    return rag_chain.invoke(question)

In [23]:
if __name__ == "__main__":
    while True:
        user_question = input(
            "\nAsk a question or type 'exit': "
        ).strip()

        if user_question.lower() == "exit":
            print("Application closed.")
            break

        try:
            answer = ask_question(user_question)

            print("\nAnswer:")
            print(answer)

        except Exception as error:
            print(f"\nError: {error}")


Answer:
RAG stands for Retrieval-Augmented Generation.

Answer:
I do not know based on the provided context.

Answer:
I do not know based on the provided context.

Answer:
I do not know based on the provided context.

Answer:
FAISS is a vector similarity-search library.

Answer:
Groq provides fast inference for supported large language models through the Groq API.
Application closed.
